<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/ml/notebooks/c5_l1.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C5-L1 · Features sin leakage
5 lags honestos + embargo. Si una feature usa el futuro, el backtest miente.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/data/c5_l1.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c5_l1.csv'), Path('data/c5_l1.csv'), Path('c5_l1.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
# 5 lags honestos: lag_k en t usa solo close de t-k
for k in range(1, 6):
    df[f'lag_{k}'] = df['close'].shift(k)
df['ret_next'] = df['close'].pct_change().shift(-1)  # etiqueta: retorno t->t+1
feat = [f'lag_{k}' for k in range(1, 6)]
data = df.dropna().reset_index(drop=True)
print(data[['dia','close']+feat+['ret_next']].head(5).to_string(index=False))
# lag_1 de cada fila debe ser el cierre del dia anterior
i = 3
assert abs(data.loc[i,'lag_1'] - df.loc[df['dia']==data.loc[i,'dia']-1,'close'].values[0]) < 1e-9, 'lag_1 debe ser el cierre anterior'
assert len(feat) == 5 and data[feat].isna().sum().sum() == 0
print('OK: 5 lags honestos, sin NaN')

In [ ]:
# Test de leakage: feature honesta vs feature con fuga (usa close futuro)
from sklearn.linear_model import Ridge, LinearRegression
X = data[feat].values; y = data['ret_next'].values
split = int(len(data)*0.7)
Xtr, Xte, ytr, yte = X[:split], X[split:], y[:split], y[split:]
r_hon = Ridge().fit(Xtr, ytr).score(Xte, yte)
Xleak = (data['close'].shift(-1)/data['close'] - 1).fillna(0).values.reshape(-1,1)  # FILTRA futuro: es la propia etiqueta
r_leak = LinearRegression().fit(Xleak[:split], ytr).score(Xleak[split:], yte)
print(f'R2 honesto={r_hon:.4f}  R2 con fuga={r_leak:.4f}')
assert r_leak > r_hon, 'la fuga debe inflar el score (eso la delata)'
assert r_leak > 0.9, 'con el futuro visto, el score debe ser casi perfecto'
print('OK: el test de desplazamiento delata el leakage')

In [ ]:
# Embargo: quita del train las ultimas filas pegadas al test
from sklearn.metrics import mean_squared_error
EMBARGO = 3
Xtr_e, ytr_e = Xtr[:-EMBARGO], ytr[:-EMBARGO]
m = Ridge().fit(Xtr_e, ytr_e)
pred = m.predict(Xte)
rmse = mean_squared_error(yte, pred) ** 0.5
print(f'train {len(Xtr_e)} (embargo={EMBARGO})  test {len(Xte)}  RMSE={rmse:.6f}')

In [ ]:
# Chequeo automatico L1
assert len(feat) == 5
assert len(Xtr_e) == len(Xtr) - EMBARGO == len(Xtr) - 3
assert np.isfinite(rmse) and rmse < 0.1
assert r_leak > r_hon
print('OK L1: 5 lags honestos + embargo verificados')